# Method 4 (CE+Dice loss) 深掘り解析

Method 4 の 5-Fold CV 結果に対する追加分析ノートブック。

## 解析内容
1. **解析1**: CE loss の効果とクラス別ピクセル分布 — 6クラスの不均衡がCE loss改善に与える影響
2. **解析2**: 楕円近似の改善効果 — raw→ellipse の改善量をMethod3/4で比較
3. **解析3**: Method1 Pupil 回帰失敗の原因分析 — サイズ依存性と遮蔽の影響
4. **解析4**: 統計的有意性検定 — 全手法間の対応t検定・効果量

## 前提
- `crossvalidation.ipynb` で Method 1-4 の 5-Fold CV が完了済み
- 結果CSVが `results/` に保存済み
- ラベル画像が `Images/labels_seg/` に存在

**作成日**: 2026-02-22  
**記録**: `Experimental_record/20260222.md`

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from PIL import Image
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# パス設定
LABELS_DIR = 'Images/labels_seg'
RESULTS_DIR = 'results'

# 結果CSVの読み込み（最新のタイムスタンプを自動検出）
def find_latest_csv(pattern):
    files = sorted(glob.glob(os.path.join(RESULTS_DIR, pattern)))
    if not files:
        raise FileNotFoundError(f'No files matching {pattern}')
    return files[-1]

m1_eval = pd.read_csv(find_latest_csv('cv_eval_method1_*.csv'))
m2_eval = pd.read_csv(find_latest_csv('cv_eval_method2_*.csv'))
m3_eval = pd.read_csv(find_latest_csv('cv_eval_method3_*.csv'))
m4_eval = pd.read_csv(find_latest_csv('cv_eval_method4_*.csv'))

m1_perimage = pd.read_csv(find_latest_csv('cv_method1_perimage_*.csv'))
m3_perimage = pd.read_csv(find_latest_csv('cv_method3_full_vs_exposed_perimage_*.csv'))
m4_perimage = pd.read_csv(find_latest_csv('cv_method4_full_vs_exposed_perimage_*.csv'))

print('Loaded files:')
print(f'  M1 eval: {m1_eval.shape}, M1 perimage: {m1_perimage.shape}')
print(f'  M2 eval: {m2_eval.shape}')
print(f'  M3 eval: {m3_eval.shape}, M3 perimage: {m3_perimage.shape}')
print(f'  M4 eval: {m4_eval.shape}, M4 perimage: {m4_perimage.shape}')

---
## 解析1: CE loss の効果とクラス別ピクセル分布

6クラスのピクセル不均衡を計測し、CE loss がどのクラスに最も効果的かを分析する。

In [ ]:
# 全画像の6クラスピクセル分布を計算
# 個別マスクファイル (pupil_vis, pupil_occ, iris_vis, iris_occ, mask_lid) から算出

image_dir = 'Images/images'
image_files = sorted(glob.glob(os.path.join(image_dir, '*.jpg')))
print(f'Total images: {len(image_files)}')

class_stats = []
for img_path in image_files:
    stem = os.path.splitext(os.path.basename(img_path))[0]
    
    masks = {}
    for region in ['pupil_vis', 'pupil_occ', 'iris_vis', 'iris_occ', 'mask_lid']:
        path = os.path.join(LABELS_DIR, f'{stem}_{region}.png')
        if os.path.exists(path):
            img = np.array(Image.open(path))
            if len(img.shape) == 3:
                img = img[:, :, 0]
            masks[region] = (img > 127).astype(bool)
        else:
            masks[region] = np.zeros((512, 512), dtype=bool)
    
    lid = masks['mask_lid']
    iris = masks['iris_vis'] | masks['iris_occ']
    pupil = masks['pupil_vis'] | masks['pupil_occ']
    
    total = 512 * 512
    pv = masks['pupil_vis'].sum()
    po = masks['pupil_occ'].sum()
    iv = masks['iris_vis'].sum()
    io = masks['iris_occ'].sum()
    conj = (lid & ~iris & ~pupil).sum()
    bg = total - conj - iv - io - pv - po
    
    class_stats.append({
        'filename': os.path.basename(img_path),
        'background': bg, 'conjunctiva': conj,
        'iris_vis': iv, 'iris_occ': io,
        'pupil_vis': pv, 'pupil_occ': po,
        'total': total
    })

stats_df = pd.DataFrame(class_stats)
print(f'Processed: {len(stats_df)} images')

In [ ]:
# クラス別分布の集計と可視化
classes = ['background', 'conjunctiva', 'iris_vis', 'iris_occ', 'pupil_vis', 'pupil_occ']
class_labels = ['Background(0)', 'Conjunctiva(1)', 'Iris vis(2)', 'Iris occ(3)', 'Pupil vis(4)', 'Pupil occ(5)']

print('=== 6-Class Pixel Distribution ===')
print(f'{"Class":20s} {"Mean Pixels":>12s} {"Mean %":>8s} {"Std %":>8s}')
print('-' * 52)
mean_pcts = []
for cls, label in zip(classes, class_labels):
    pct = stats_df[cls] / stats_df['total'] * 100
    mean_pcts.append(pct.mean())
    print(f'{label:20s} {stats_df[cls].mean():12.0f} {pct.mean():7.2f}% {pct.std():7.2f}%')

print(f'\nMax/min class ratio: {max(mean_pcts)/max(min(mean_pcts), 0.001):.0f}:1')
print(f'Iris vis/occ ratio: {stats_df["iris_vis"].mean()/max(stats_df["iris_occ"].mean(), 1):.1f}:1')
print(f'Pupil vis/occ ratio: {stats_df["pupil_vis"].mean()/max(stats_df["pupil_occ"].mean(), 1):.1f}:1')

# 棒グラフ
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 全クラス（対数スケール）
ax = axes[0]
bars = ax.bar(class_labels, mean_pcts, color=['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0', '#795548'])
ax.set_ylabel('Mean pixel fraction (%)')
ax.set_title('6-Class Pixel Distribution (log scale)')
ax.set_yscale('log')
ax.set_ylim(0.01, 100)
for bar, pct in zip(bars, mean_pcts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.2, f'{pct:.2f}%', 
            ha='center', va='bottom', fontsize=8)
ax.tick_params(axis='x', rotation=30)

# Background除外（線形スケール）
ax = axes[1]
bars = ax.bar(class_labels[1:], mean_pcts[1:], color=['#4CAF50', '#FF9800', '#F44336', '#9C27B0', '#795548'])
ax.set_ylabel('Mean pixel fraction (%)')
ax.set_title('Foreground Classes Only (linear scale)')
for bar, pct in zip(bars, mean_pcts[1:]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, f'{pct:.2f}%', 
            ha='center', va='bottom', fontsize=9)
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('results/analysis1_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/analysis1_class_distribution.png')

### 解析1 考察

- Background が 86.5% を占め、pupil_occ は僅か 0.03%（**2501:1 の不均衡**）
- CE loss はピクセル単位で各クラスを等しく扱うため、Dice-only loss より小クラス（iris_occ, pupil_occ）の境界学習を改善
- Iris occluded（2.08%）は CE loss の恩恵を最も受けやすい → Iris で最大改善（+0.0235）

---
## 解析2: 楕円近似の改善効果（Method3 vs Method4）

M3→M4 の改善が「生セグメンテーション品質」と「楕円近似の後処理」のどちらに由来するかを分離する。

In [ ]:
# 楕円近似手法の比較
print('=== Ellipse Fitting Mode Comparison ===')
print(f'{"":15s} {"Method3 Iris":>12s} {"M3 Pupil":>10s} {"Method4 Iris":>12s} {"M4 Pupil":>10s}')
print('-' * 63)
for mode in ['raw', 'outerarc', 'fullmax', 'ransac_whole']:
    m3_sub = m3_perimage[m3_perimage['mode'] == mode]
    m4_sub = m4_perimage[m4_perimage['mode'] == mode]
    marker = ' <-- best' if mode == 'fullmax' else ''
    print(f'{mode:15s} {m3_sub["iris"].mean():12.4f} {m3_sub["pupil"].mean():10.4f} '
          f'{m4_sub["iris"].mean():12.4f} {m4_sub["pupil"].mean():10.4f}{marker}')

In [ ]:
# 楕円近似による改善量 (fullmax - raw)
print('=== Ellipse Fitting Improvement (fullmax - raw) ===')
for name, df in [('Method3', m3_perimage), ('Method4', m4_perimage)]:
    raw = df[df['mode'] == 'raw'][['filename', 'iris', 'pupil']].set_index('filename')
    best = df[df['mode'] == 'fullmax'][['filename', 'iris', 'pupil']].set_index('filename')
    
    iris_improve = best['iris'] - raw['iris']
    pupil_improve = best['pupil'] - raw['pupil']
    
    print(f'\n{name}:')
    print(f'  Iris:  +{iris_improve.mean():.4f} +/- {iris_improve.std():.4f}')
    print(f'    improved: {(iris_improve > 0).sum()}/{len(iris_improve)} ({(iris_improve > 0).mean()*100:.1f}%)')
    print(f'    worsened (>0.01): {(iris_improve < -0.01).sum()}/{len(iris_improve)} ({(iris_improve < -0.01).mean()*100:.1f}%)')
    print(f'  Pupil: {pupil_improve.mean():+.4f} +/- {pupil_improve.std():.4f}')
    print(f'    improved: {(pupil_improve > 0).sum()}/{len(pupil_improve)} ({(pupil_improve > 0).mean()*100:.1f}%)')
    print(f'    worsened (>0.01): {(pupil_improve < -0.01).sum()}/{len(pupil_improve)} ({(pupil_improve < -0.01).mean()*100:.1f}%)')

In [ ]:
# M3→M4 改善の分解: 生セグメンテーション vs 楕円近似
m3_raw = m3_perimage[m3_perimage['mode'] == 'raw'][['filename', 'iris', 'pupil']].set_index('filename')
m4_raw = m4_perimage[m4_perimage['mode'] == 'raw'][['filename', 'iris', 'pupil']].set_index('filename')
m3_fm = m3_perimage[m3_perimage['mode'] == 'fullmax'][['filename', 'iris', 'pupil']].set_index('filename')
m4_fm = m4_perimage[m4_perimage['mode'] == 'fullmax'][['filename', 'iris', 'pupil']].set_index('filename')

print('=== Decomposition of M3 -> M4 Improvement ===')
for structure in ['iris', 'pupil']:
    total = m4_fm[structure].mean() - m3_fm[structure].mean()
    raw_gain = m4_raw[structure].mean() - m3_raw[structure].mean()
    ell_diff = total - raw_gain
    
    m3_ell = m3_fm[structure].mean() - m3_raw[structure].mean()
    m4_ell = m4_fm[structure].mean() - m4_raw[structure].mean()
    
    print(f'\n{structure.upper()}:')
    print(f'  Total improvement (fullmax):     {total:+.4f}')
    print(f'  Raw segmentation improvement:    {raw_gain:+.4f} ({raw_gain/total*100:.1f}%)')
    print(f'  Ellipse fitting difference:      {ell_diff:+.4f} ({ell_diff/total*100:.1f}%)')
    print(f'  M3 ellipse gain: {m3_ell:+.4f}, M4 ellipse gain: {m4_ell:+.4f}')

In [ ]:
# 失敗画像数 (Dice < 0.7) の比較
print('=== Failure Cases (Dice < 0.7) ===')
print(f'{"Method/Mode":20s} {"Iris<0.7":>10s} {"Pupil<0.7":>10s}')
print('-' * 44)
for name, df in [('Method3', m3_perimage), ('Method4', m4_perimage)]:
    for mode in ['raw', 'fullmax']:
        sub = df[df['mode'] == mode]
        iris_fail = (sub['iris'] < 0.7).sum()
        pupil_fail = (sub['pupil'] < 0.7).sum()
        label = f'{name} {mode}'
        print(f'{label:20s} {iris_fail:4d} ({iris_fail/len(sub)*100:.1f}%) {pupil_fail:4d} ({pupil_fail/len(sub)*100:.1f}%)')

# M4がM3より大幅に劣化する画像
iris_diff = m4_fm['iris'] - m3_fm['iris']
print(f'\n=== Images where M4 iris worse than M3 by >0.05: {(iris_diff < -0.05).sum()} ===')
for fn in iris_diff.nsmallest(5).index:
    print(f'  {fn[:70]}  M3={m3_fm.loc[fn, "iris"]:.4f} M4={m4_fm.loc[fn, "iris"]:.4f} diff={iris_diff[fn]:+.4f}')

In [ ]:
# 可視化: M3 vs M4 改善の散布図
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, structure in zip(axes, ['iris', 'pupil']):
    common = m3_fm.index.intersection(m4_fm.index)
    x = m3_fm.loc[common, structure]
    y = m4_fm.loc[common, structure]
    
    ax.scatter(x, y, alpha=0.15, s=8, c='steelblue')
    ax.plot([0, 1], [0, 1], 'r--', alpha=0.5, label='y=x')
    ax.set_xlabel(f'Method3 {structure} Dice (fullmax)')
    ax.set_ylabel(f'Method4 {structure} Dice (fullmax)')
    ax.set_title(f'{structure.capitalize()}: M3 vs M4 (n={len(common)})')
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.02)
    ax.legend()
    
    above = (y > x).sum()
    ax.text(0.05, 0.92, f'M4 better: {above}/{len(common)} ({above/len(common)*100:.1f}%)',
            transform=ax.transAxes, fontsize=10)

plt.tight_layout()
plt.savefig('results/analysis2_m3_vs_m4_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/analysis2_m3_vs_m4_scatter.png')

### 解析2 考察

- M3→M4 の改善は**ほぼ100%が生セグメンテーション品質の向上に由来**（楕円近似の貢献はほぼゼロ）
- 楕円近似は Iris に劇的効果（+0.067）だが Pupil にはほぼ無効（元々円形）
- CE loss が Iris raw failure を 119→33 に **72% 削減**

---
## 解析3: Method1 Pupil 回帰失敗の原因分析

Method1 の Pupil Dice が 0.7225 と低い原因を、サイズ依存性と遮蔽の観点から分析する。

In [ ]:
# Method1 Pupil Dice 分布
print('=== Method1 Pupil Dice Distribution ===')
print(f'Mean: {m1_perimage["pupil"].mean():.4f}, Std: {m1_perimage["pupil"].std():.4f}')
print(f'Iris Mean: {m1_perimage["iris"].mean():.4f}, Gap: {m1_perimage["iris"].mean() - m1_perimage["pupil"].mean():.4f}')
print()

bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.01]
labels = ['0-0.1', '0.1-0.2', '0.2-0.3', '0.3-0.4', '0.4-0.5',
          '0.5-0.6', '0.6-0.7', '0.7-0.8', '0.8-0.9', '0.9-0.95', '0.95-1.0']
m1_perimage['pupil_bin'] = pd.cut(m1_perimage['pupil'], bins=bins, labels=labels, right=False)
dist = m1_perimage['pupil_bin'].value_counts().sort_index()

for label, count in dist.items():
    pct = count / len(m1_perimage) * 100
    bar = '#' * int(pct)
    print(f'  {label:10s}: {count:4d} ({pct:5.1f}%) {bar}')

print(f'\nTotal failures (< 0.5): {(m1_perimage["pupil"] < 0.5).sum()} / {len(m1_perimage)} '
      f'({(m1_perimage["pupil"] < 0.5).mean()*100:.1f}%)')

In [ ]:
# Pupilサイズの計測とMethod1 Diceとの相関
pupil_sizes = []
for fn in m1_perimage['filename']:
    stem = os.path.splitext(fn)[0]
    pv = po = 0
    pv_path = os.path.join(LABELS_DIR, f'{stem}_pupil_vis.png')
    po_path = os.path.join(LABELS_DIR, f'{stem}_pupil_occ.png')
    if os.path.exists(pv_path):
        img = np.array(Image.open(pv_path))
        if len(img.shape) == 3: img = img[:, :, 0]
        pv = (img > 127).sum()
    if os.path.exists(po_path):
        img = np.array(Image.open(po_path))
        if len(img.shape) == 3: img = img[:, :, 0]
        po = (img > 127).sum()
    pupil_sizes.append({'pupil_total': pv + po, 'pupil_vis': pv, 'pupil_occ': po})

sizes_df = pd.DataFrame(pupil_sizes)
m1_analysis = m1_perimage.copy()
m1_analysis['pupil_total'] = sizes_df['pupil_total'].values
m1_analysis['pupil_vis_frac'] = sizes_df['pupil_vis'].values / sizes_df['pupil_total'].clip(lower=1).values
m1_analysis['pupil_ratio'] = m1_analysis['pupil_total'] / (512 * 512)

print('=== Correlation Analysis ===')
print(f'Pupil size vs Pupil Dice: r = {m1_analysis["pupil_total"].corr(m1_analysis["pupil"]):.4f}')
print(f'Pupil vis fraction vs Pupil Dice: r = {m1_analysis["pupil_vis_frac"].corr(m1_analysis["pupil"]):.4f}')
print(f'\nMean pupil size: {m1_analysis["pupil_total"].mean():.0f} pixels ({m1_analysis["pupil_ratio"].mean()*100:.2f}% of 512x512)')

In [ ]:
# サイズ四分位別の Method1 vs Method4 比較
m4_fm_for_merge = m4_perimage[m4_perimage['mode'] == 'fullmax'][['filename', 'pupil']].rename(
    columns={'pupil': 'pupil_m4'})
merged = m1_analysis.merge(m4_fm_for_merge, on='filename', how='inner')

merged['size_q'] = pd.qcut(merged['pupil_total'], 4, labels=['Q1(tiny)', 'Q2', 'Q3', 'Q4(large)'])

print('=== Method1 vs Method4 by Pupil Size Quartile ===')
print(f'{"Quartile":12s} {"Avg pixels":>10s} {"M1 Dice":>9s} {"M4 Dice":>9s} {"Diff":>8s} {"M1 fail":>8s} {"M4 fail":>8s}')
print('-' * 72)
for q in ['Q1(tiny)', 'Q2', 'Q3', 'Q4(large)']:
    sub = merged[merged['size_q'] == q]
    m1_fail = (sub['pupil'] < 0.5).sum()
    m4_fail = (sub['pupil_m4'] < 0.5).sum()
    print(f'{q:12s} {sub["pupil_total"].mean():10.0f} {sub["pupil"].mean():9.4f} '
          f'{sub["pupil_m4"].mean():9.4f} {sub["pupil_m4"].mean()-sub["pupil"].mean():+8.4f} '
          f'{m1_fail:8d} {m4_fail:8d}')

In [ ]:
# M1失敗画像でのM4性能
m1_fail_mask = merged['pupil'] < 0.5
m1_failures = merged[m1_fail_mask]

print(f'=== M1 Failure Images (pupil < 0.5): {len(m1_failures)} ===' )
print(f'M4 on these images: {m1_failures["pupil_m4"].mean():.4f} +/- {m1_failures["pupil_m4"].std():.4f}')
print(f'M4 also fails (<0.5): {(m1_failures["pupil_m4"] < 0.5).sum()}/{len(m1_failures)} ({(m1_failures["pupil_m4"] < 0.5).mean()*100:.1f}%)')
print(f'M4 recovers (>0.7):  {(m1_failures["pupil_m4"] > 0.7).sum()}/{len(m1_failures)} ({(m1_failures["pupil_m4"] > 0.7).mean()*100:.1f}%)')
print(f'M4 excellent (>0.9): {(m1_failures["pupil_m4"] > 0.9).sum()}/{len(m1_failures)} ({(m1_failures["pupil_m4"] > 0.9).mean()*100:.1f}%)')

# 遮蔽の影響
print(f'\n=== Pupil Dice by Visibility Fraction ===')
vis_bins = [0, 0.5, 0.7, 0.85, 1.01]
vis_labels = ['0-50% (high occ)', '50-70%', '70-85%', '85-100% (low occ)']
m1_analysis['vis_bin'] = pd.cut(m1_analysis['pupil_vis_frac'], bins=vis_bins, labels=vis_labels, right=False)
for b in vis_labels:
    sub = m1_analysis[m1_analysis['vis_bin'] == b]
    if len(sub) > 0:
        fail_rate = (sub['pupil'] < 0.5).mean() * 100
        print(f'  {b:20s}: n={len(sub):4d}, Dice={sub["pupil"].mean():.4f}, fail rate={fail_rate:.1f}%')

In [ ]:
# 可視化: サイズ依存性と手法比較
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. M1 Pupil Dice ヒストグラム
ax = axes[0]
ax.hist(m1_perimage['pupil'], bins=50, alpha=0.7, color='steelblue', edgecolor='white')
ax.axvline(0.5, color='red', linestyle='--', alpha=0.7, label='fail threshold (0.5)')
ax.set_xlabel('Method1 Pupil Dice')
ax.set_ylabel('Count')
ax.set_title(f'M1 Pupil Distribution (mean={m1_perimage["pupil"].mean():.3f})')
ax.legend()

# 2. Pupil size vs Dice scatter
ax = axes[1]
ax.scatter(m1_analysis['pupil_total'], m1_analysis['pupil'], alpha=0.15, s=8, c='steelblue')
ax.set_xlabel('Pupil size (pixels)')
ax.set_ylabel('Method1 Pupil Dice')
ax.set_title(f'Size vs Dice (r={m1_analysis["pupil_total"].corr(m1_analysis["pupil"]):.3f})')
ax.axhline(0.5, color='red', linestyle='--', alpha=0.5)

# 3. M1 vs M4 by size quartile
ax = axes[2]
quartiles = ['Q1(tiny)', 'Q2', 'Q3', 'Q4(large)']
m1_means = [merged[merged['size_q'] == q]['pupil'].mean() for q in quartiles]
m4_means = [merged[merged['size_q'] == q]['pupil_m4'].mean() for q in quartiles]
x = np.arange(len(quartiles))
width = 0.35
bars1 = ax.bar(x - width/2, m1_means, width, label='Method1 (regression)', color='#F44336', alpha=0.8)
bars2 = ax.bar(x + width/2, m4_means, width, label='Method4 (segmentation)', color='#2196F3', alpha=0.8)
ax.set_xlabel('Pupil Size Quartile')
ax.set_ylabel('Pupil Dice')
ax.set_title('Regression vs Segmentation by Pupil Size')
ax.set_xticks(x)
ax.set_xticklabels(quartiles)
ax.legend()
ax.set_ylim(0.4, 1.0)

plt.tight_layout()
plt.savefig('results/analysis3_m1_pupil_failure.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/analysis3_m1_pupil_failure.png')

### 解析3 考察

- 回帰アプローチの**サイズ依存性**が根本原因: 最小四分位で M1=0.584 vs M4=0.906（差 0.32）
- 瞼遮蔽率50%超で fail rate **64.3%** — 回帰は見えない部分の補間が困難
- M1 失敗画像の **80.1%** を M4 が回復

---
## 解析4: 統計的有意性検定（全手法間比較）

In [ ]:
# Fold レベル対応 t 検定: Method3 vs Method4
print('=== Paired t-test: Method3 vs Method4 (n=5 folds) ===')
print(f'{"Metric":8s} {"Diff":>8s} {"t":>8s} {"p-value":>10s} {"Sig":>5s}')
print('-' * 45)
for metric in ['eyelid', 'iris', 'pupil', 'mean']:
    t_stat, p_value = stats.ttest_rel(m4_eval[metric], m3_eval[metric])
    diff = m4_eval[metric].mean() - m3_eval[metric].mean()
    sig = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'n.s.'
    print(f'{metric:8s} {diff:+8.4f} {t_stat:8.3f} {p_value:10.4f} {sig:>5s}')

In [ ]:
# 全手法ペア比較 (Mean Dice)
print('=== All Pairwise Paired t-tests (Mean Dice, n=5 folds) ===')
methods = {'M1': m1_eval, 'M2': m2_eval, 'M3': m3_eval, 'M4': m4_eval}
print(f'{"Comparison":12s} {"Diff":>8s} {"t":>8s} {"p-value":>10s} {"Sig":>5s}')
print('-' * 48)
method_items = list(methods.items())
for i, (n1, d1) in enumerate(method_items):
    for n2, d2 in method_items[i+1:]:
        t_stat, p_value = stats.ttest_rel(d2['mean'], d1['mean'])
        diff = d2['mean'].mean() - d1['mean'].mean()
        sig = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'n.s.'
        print(f'{n1} vs {n2:4s} {diff:+8.4f} {t_stat:8.3f} {p_value:10.4f} {sig:>5s}')

In [ ]:
# 画像レベル対応 t 検定 + Cohen's d (Method3 vs Method4)
print('=== Per-image Paired t-test: Method3 vs Method4 (fullmax, n=1992) ===')
common = m3_fm.index.intersection(m4_fm.index)
print(f'{"Metric":8s} {"Diff":>8s} {"t":>10s} {"p-value":>12s} {"Cohen d":>9s}')
print('-' * 52)
for metric in ['iris', 'pupil', 'mean']:
    diff = m4_fm.loc[common, metric] - m3_fm.loc[common, metric]
    t_stat, p_value = stats.ttest_rel(m4_fm.loc[common, metric], m3_fm.loc[common, metric])
    d = diff.mean() / diff.std()  # Cohen's d
    print(f'{metric:8s} {diff.mean():+8.4f} {t_stat:10.3f} {p_value:12.2e} {d:9.3f}')

In [ ]:
# Wilcoxon signed-rank test (非パラメトリック)
print('=== Wilcoxon Signed-Rank: Method3 vs Method4 (n=5 folds) ===')
print('Note: n=5 limits detection power (minimum possible p = 0.0625)')
print()
for metric in ['eyelid', 'iris', 'pupil', 'mean']:
    diff = m4_eval[metric] - m3_eval[metric]
    all_positive = (diff > 0).all()
    try:
        w_stat, p_value = stats.wilcoxon(diff)
        print(f'  {metric:8s}: W={w_stat:.1f}, p={p_value:.4f}, all folds positive: {all_positive}')
    except Exception as e:
        print(f'  {metric:8s}: {e}')

In [ ]:
# 可視化: 全手法比較
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Fold-level Mean Dice boxplot
ax = axes[0]
data = [m1_eval['mean'], m2_eval['mean'], m3_eval['mean'], m4_eval['mean']]
bp = ax.boxplot(data, labels=['M1\n(Regression)', 'M2\n(Edge)', 'M3\n(6cls Dice)', 'M4\n(6cls CE+Dice)'],
                patch_artist=True)
colors = ['#F44336', '#FF9800', '#4CAF50', '#2196F3']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_ylabel('Mean Dice')
ax.set_title('5-Fold CV Mean Dice by Method')

# Add significance bars
y_max = max(d.max() for d in data) + 0.005
ax.plot([3, 4], [y_max + 0.003, y_max + 0.003], 'k-', linewidth=1)
ax.text(3.5, y_max + 0.004, '**p=0.001', ha='center', fontsize=8)

# 2. Per-structure comparison
ax = axes[1]
structures = ['Eyelid', 'Iris', 'Pupil', 'Mean']
x = np.arange(len(structures))
width = 0.2
for i, (name, df, color) in enumerate([
    ('M1', m1_eval, '#F44336'), ('M2', m2_eval, '#FF9800'),
    ('M3', m3_eval, '#4CAF50'), ('M4', m4_eval, '#2196F3')
]):
    means = [df['eyelid'].mean(), df['iris'].mean(), df['pupil'].mean(), df['mean'].mean()]
    stds = [df['eyelid'].std(), df['iris'].std(), df['pupil'].std(), df['mean'].std()]
    ax.bar(x + i * width, means, width, yerr=stds, label=name, color=color, alpha=0.8, capsize=3)

ax.set_ylabel('Dice Coefficient')
ax.set_title('Per-Structure Performance')
ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(structures)
ax.legend()
ax.set_ylim(0.6, 1.0)

plt.tight_layout()
plt.savefig('results/analysis4_statistical_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/analysis4_statistical_comparison.png')

### 解析4 考察

- M3 vs M4: 全指標で有意（Iris: p=0.0003***, Mean: p=0.0011**）
- 全6ペア比較で有意差 → **M1 < M2 < M3 < M4** の順序が統計的に確立
- Cohen's d: Iris 0.425（小〜中）、Mean 0.383（小〜中）— 実質的にも意味のある効果
- Wilcoxon 検定は n=5 で検出力不足（p=0.0625が下限）

---
## 全体まとめ

| 解析 | 主要発見 |
|------|--------|
| 1. クラス不均衡 | 2501:1の不均衡。CE loss が小クラス（iris_occ）の学習を改善 |
| 2. 楕円近似分解 | M3→M4 改善の100%が生セグメンテーション由来。楕円近似の貢献はゼロ |
| 3. M1 Pupil 失敗 | 回帰のサイズ依存性が原因。最小Qで M1=0.58 vs M4=0.91。M1失敗の80%をM4が回復 |
| 4. 統計的有意性 | 全手法間で有意差確立。M4は全指標でM3に有意に優越 (p<0.05) |